In [21]:
from concurrent.futures import ThreadPoolExecutor, as_completed
import datetime
from random import randrange
import time

import duckdb
import hdbscan
import joblib
import pandas as pd
import requests
import xgboost as xgb

CATMODEL = joblib.load("hdbscan_model.pkl")
SCALER = joblib.load("scaler.pkl")

def getmarket(n:int):
    time = datetime.datetime.fromtimestamp(1758244600).isoformat()
    url = "https://gamma-api.polymarket.com/markets"
    response = requests.get(
        url,
        params={
            "limit":n,
            "volume_num_min":100000,
            "start_date_min":f'{time+'Z'}',
            "closed":"true",
            "offset":100,
            "include_tag":True

        }
    )
    return response.json()

def tradesByMarket(condition_id):


    nonTrivialTrades = []
    offset = 0
    while len(nonTrivialTrades) < 1000:
        url = "https://data-api.polymarket.com/trades"
        response = requests.get(
            url,
            params = {
                "limit":"1000",
                "market":condition_id,
                "side":"BUY",
                "offset": offset,
                "filterType":"CASH",
                "filterAmount":5
            }
        )
        trades = response.json()
        filteredTrades = [t for t in trades if 0.2<t['price']<0.9] 
        nonTrivialTrades.extend(filteredTrades)
        offset += 1000
        if offset == 4000: #Offset limit is 3000
            break

    return nonTrivialTrades
def stringToList(string):
    listOf = string.split(',')
    listOf[0] = listOf[0][2:-1]
    listOf[1] = listOf[1][2:-2]
    return listOf
def did_win(row):
    try:
        outcomes = row['outcomes']    
        prices = row['outcomePrices']    
        idx = outcomes.index(row['outcome'])
        return float(prices[idx]) >= 0.85
    except (ValueError, IndexError):
        return False
def get_end_date(m):
    if 'endDate' in m and m['endDate']:
        return m['endDate']
    if m.get('events') and len(m['events']) > 0 and m['events'][0].get('endDate'):
        return m['events'][0]['endDate']
    if m.get('closedTime'):
        return m['closedTime']
    if 'umaEndDate' in m and m['umaEndDate']:
        return m['umaEndDate']
    return KeyError
def get_start_date(m):
    if m.get('startDate'):
        return m['startDate']
    if m.get('events') and len(m['events']) > 0 and m['events'][0].get('startDate'):
        return m['events'][0]['startDate']
    if m.get('createdAt'):
        return m['createdAt']
    if m.get('acceptingOrdersTimestamp'):
        return m['acceptingOrdersTimestamp']
    return KeyError
def timeSinceLastTrade(df):
    df = df.sort_values('timestamp').copy()
    df['prev_timestamp'] = df.groupby(['proxyWallet'])['timestamp'].shift(1)
    df['refTime'] = df['prev_timestamp'].fillna(df['last_trade'])

    df['time_since_last_trade'] = df['timestamp'] - df['refTime']
    return df
def getUserData(u):
    offset = 0
    userTrades=[]
    while True:
        url = "https://data-api.polymarket.com/trades"
        response = requests.get(
            url,
            params={
                "limit":1000,
                "offset":offset,
                "filterType":"CASH",
                "filterAmount":2,
                "user":u,
                "side":"BUY"
            }
        )
        userTrades.extend(response.json())
        if len(userTrades) < offset+1000 or offset == 3000:
            break
        offset += 1000
    if len(userTrades) == 0:
        return
    try:
        uTradesDf = pd.DataFrame(userTrades)[[
            'side','size','price','timestamp','outcome','conditionId']]
    except ValueError:
        print(userTrades)
    except KeyError:
        return 
    uMarkets = list(set(uTradesDf['conditionId'].to_list()))
    offset = 0
    uMarketList = []
    while offset < len(uMarkets):
        time.sleep(0.2) # Give the api some rest
        url = "https://gamma-api.polymarket.com/markets"

        response = requests.get(
            url,
            params={
                "condition_ids":uMarkets[offset:offset+50],
                "closed":"true",
            }
        )
        try:
            for m in response.json():
                try:
                    marketdetails = [m['conditionId'],stringToList(m['outcomes']),stringToList(m['outcomePrices']),get_start_date(m),get_end_date(m)]
                    uMarketList.append(marketdetails)
                except KeyError as e:
                    print(m)
                    print(e)
                    exit()
                    continue
            offset += 50
        except requests.exceptions.JSONDecodeError:
            print(response)
    uMarketDf = pd.DataFrame(uMarketList, columns=['conditionId', 'outcomes', 'outcomePrices', 'startDate', 'endDate'])
    #trade_ids = set(uTradesDf['conditionId'].dropna())
    #market_ids = set(uMarketDf['conditionId'].dropna())

    #print(f"Unique conditionIds missed: {len(trade_ids)-len(market_ids)}")
    if len(uTradesDf) == 0:
        print("no trades")
        exit()
    uTradesDf = uTradesDf.merge(uMarketDf, on='conditionId', how='left')
    uTradesDf = uTradesDf.dropna()
    uTradesDf = uTradesDf[uTradesDf['conditionId']!=CONDITION_ID] # Removes trades from market thats looked at
    uTradesDf = uTradesDf[uTradesDf['timestamp'] < datetime.datetime.fromisoformat(end_date).timestamp()] # Removes trades that are too late / in the future (might need to find a diff cutoff)
    if len(uTradesDf) == 0:                                                                               # And kinda makes the condition filter obselete
        return
    uTradesDf['won'] = uTradesDf.apply(did_win, axis=1).astype(int)
    uTradesDf = uTradesDf.sort_values(by=["timestamp"],ascending=True)

    MainFeatures = {}
    MainFeatures['proxyWallet'] = u
    MainFeatures['avg_price'] = uTradesDf['price'].mean()
    MainFeatures['highest_price_yet'] = uTradesDf['price'].max()
    MainFeatures['lowest_price_yet'] = uTradesDf['price'].min()
    MainFeatures['avg_spent'] = uTradesDf['size'].mean()
    MainFeatures['max_spent'] = uTradesDf['size'].max()
    MainFeatures['nMarkets'] = uTradesDf['conditionId'].nunique()
    MainFeatures['nTrades'] = len(uTradesDf)
    MainFeatures['total_spent'] = uTradesDf['size'].sum()
    MainFeatures['win_rate'] = uTradesDf['won'].sum()/MainFeatures['nTrades']
    MainFeatures['last_trade'] = uTradesDf['timestamp'].iloc[-1]
    if randrange(30) == 7:
        print("peekaboo")
    return MainFeatures

def collectUserDataThreads(userList,threads:int):
    results = []
    with ThreadPoolExecutor(max_workers=threads) as executor:
        futures = {
            executor.submit(getUserData, u): u
            for u in userList
        }
        for future in as_completed(futures):
            result = future.result()
            if result is not None: 
                results.append(result)
    return results
def getPrediction(X_df,model,features,diff=""):
    X_data = xgb.DMatrix(X_df[features].to_numpy())
    prediction = model.predict(X_data)
    decision = (prediction > 0.6).astype(int)
    X_df['prediction'+diff] = decision
    return X_df
def LOOKFORMARKET(markets):
    i = 16
    try:
        while True:
            SELECTEDMARKET = markets[i]
            CONDITION_ID = SELECTEDMARKET['conditionId']
            return SELECTEDMARKET
            search = duckdb.query(f"""SELECT * FROM '{MARKETSTRAINED}' WHERE condition_id = '{CONDITION_ID}' """)
            if len(search) >= 1:
                i += 1
                print("Market skipped")
                continue
            return SELECTEDMARKET
    except IndexError:
        return None
#### BUNCH OF FUNCS YOU CAN IGNORE
##########################################################
MARKETSTRAINED = "x:/PolymarketData/markets.parquet"
markets = getmarket(100)
SELECTEDMARKET = LOOKFORMARKET(markets)
print(SELECTEDMARKET['slug'])
CONDITION_ID = SELECTEDMARKET['conditionId']
end_date = get_end_date(SELECTEDMARKET)
start_date = get_start_date(SELECTEDMARKET)

print(SELECTEDMARKET)
print(SELECTEDMARKET.keys())
print(SELECTEDMARKET['tags'])

fl1-met-olm-2025-10-04-olm
{'id': '606112', 'question': 'Will Olympique de Marseille win on 2025-10-04?', 'conditionId': '0x36a22a727d60714a764e1b238b3ec47cd6010ded3e5dba45b4a458de3d7f2a3f', 'slug': 'fl1-met-olm-2025-10-04-olm', 'resolutionSource': 'https://ligue1.com/en', 'endDate': '2025-10-04T15:00:00Z', 'startDate': '2025-09-21T04:04:57.755541Z', 'fee': '20000000000000000', 'image': 'https://polymarket-upload.s3.us-east-2.amazonaws.com/team_logos/fl1/olm.png', 'icon': 'https://polymarket-upload.s3.us-east-2.amazonaws.com/team_logos/fl1/olm.png', 'description': 'In the upcoming FL1 game, scheduled for October 4 at 11:00AM ET,\nIf Olympique de Marseille wins, this market will resolve to “Yes”.\nIf Olympique de Marseille loses, this market will resolve to “No”.\nIf the game is postponed, this market will remain open until the game has been completed.\nIf the game is canceled entirely, with no make-up game, this market will resolve “No”.\nThis market refers only to the outcome within t

In [5]:
print(predictedDfs.head(1))

    time_since_start  time_until_end  hour_of_day  day_of_week   winrate  \
56     769595.244459        392907.0            1            1  0.517056   

    cum_spent_prior  cum_max_spent  highest_price_yet  lowest_price_yet  \
56     37211.392733    1282.051281               0.99          0.019671   

    avg_price  ...  usd_amount  token_amount  price  direction  Group  \
56   0.543487  ...   30.769228     19.999998   0.65          1     -1   

    outcome  prediction predictionold  correctPredict  correctPredictOLD  
56      Yes         1.0           1.0            True               True  

[1 rows x 22 columns]
